# Sightline — Environment Setup
Run this notebook once before running any other Sightline notebooks.
Creates the Hive database, tests PostgreSQL connectivity, and verifies GBIF access.

In [0]:
import psycopg2
import requests

In [0]:
dbutils.widgets.text("postgres_password", "", "PostgreSQL Password")
postgres_password = dbutils.widgets.get("postgres_password")
print(f"Password length: {len(postgres_password)}")

Password length: 11


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_weather_jm1535.sightline")
spark.sql("USE dbw_weather_jm1535.sightline")
print("SUCCESS  schema 'dbw_weather_jm1535.sightline' ready")

SUCCESS  schema 'dbw_weather_jm1535.sightline' ready


In [0]:
import psycopg2

postgres_password = dbutils.widgets.get("postgres_password")

_pg_dsn = (
    "host=psql-sightline-dev.postgres.database.azure.com "
    "port=5432 "
    "dbname=sightline "
    "user=sightline_admin "
    f"password={postgres_password} "
    "sslmode=require"
)

try:
    with psycopg2.connect(_pg_dsn) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM sightings")
            (sightings_count,) = cur.fetchone()
    print(f"SUCCESS  PostgreSQL connected — {sightings_count:,} sightings in DB")
except Exception as exc:
    print(f"FAILURE  PostgreSQL connection failed: {exc}")
    raise

SUCCESS  PostgreSQL connected — 10,001 sightings in DB


In [0]:
import requests

try:
    resp = requests.get(
        "https://api.gbif.org/v1/occurrence/search",
        params={"country": "AU", "limit": 1},
        timeout=10,
    )
    resp.raise_for_status()
    total = resp.json().get("count", "unknown")
    print(f"SUCCESS  GBIF API reachable — {total:,} Australian occurrences available")
except Exception as exc:
    print(f"FAILURE  GBIF API unreachable: {exc}")
    raise

SUCCESS  GBIF API reachable — 256,419,145 Australian occurrences available


In [0]:
import json
from datetime import datetime

# Fetch Australian bird occurrences from GBIF
# Filtering to: birds (class=Aves), Australia, last 5 years, with coordinates
GBIF_URL = "https://api.gbif.org/v1/occurrence/search"
BATCH_SIZE = 300
MAX_RECORDS = 50000

params = {
    "country": "AU",
    "classKey": 212,        # Aves (birds)
    "hasCoordinate": "true",
    "hasGeospatialIssue": "false",
    "year": "2020,2026",
    "limit": BATCH_SIZE,
    "offset": 0,
}

records = []
while len(records) < MAX_RECORDS:
    resp = requests.get(GBIF_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    
    batch = data.get("results", [])
    if not batch:
        break
        
    records.extend(batch)
    params["offset"] += BATCH_SIZE
    
    if data.get("endOfRecords", True):
        break
    
    if len(records) % 3000 == 0:
        print(f"Fetched {len(records):,} records...")

print(f"Total fetched: {len(records):,} records")
print(f"Sample record keys: {list(records[0].keys())[:10]}")

Fetched 3,000 records...
Fetched 6,000 records...
Fetched 9,000 records...
Fetched 12,000 records...


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Read sightings + species from PostgreSQL into Spark
jdbc_url = (
    f"jdbc:postgresql://psql-sightline-dev.postgres.database.azure.com:5432/sightline"
    f"?sslmode=require"
)

jdbc_properties = {
    "user": "sightline_admin",
    "password": postgres_password,
    "driver": "org.postgresql.Driver"
}

# Read sightings with species name joined
sightings_df = spark.read.jdbc(
    url=jdbc_url,
    table="""(
        SELECT 
            s.id,
            s.observed_at,
            s.count,
            s.behaviour_notes,
            s.visibility,
            s.verified,
            ST_Y(s.geometry) as latitude,
            ST_X(s.geometry) as longitude,
            sp.common_name as species_common_name,
            sp.scientific_name as species_scientific_name,
            sp.kingdom,
            sp.family,
            sp.conservation_status
        FROM sightings s
        JOIN species sp ON s.species_id = sp.id
        WHERE s.visibility = 'public'
    ) as sightings_joined""",
    properties=jdbc_properties
)

print(f"Loaded {sightings_df.count():,} sightings")
sightings_df.printSchema()

Loaded 7,990 sightings
root
 |-- id: string (nullable = true)
 |-- observed_at: timestamp (nullable = true)
 |-- count: integer (nullable = true)
 |-- behaviour_notes: string (nullable = true)
 |-- visibility: string (nullable = true)
 |-- verified: boolean (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- species_common_name: string (nullable = true)
 |-- species_scientific_name: string (nullable = true)
 |-- kingdom: string (nullable = true)
 |-- family: string (nullable = true)
 |-- conservation_status: string (nullable = true)



In [0]:
# Write to Delta table
sightings_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_weather_jm1535.sightline.canonical_sightings")

print("SUCCESS  written to dbw_weather_jm1535.sightline.canonical_sightings")

# Verify
count = spark.table("dbw_weather_jm1535.sightline.canonical_sightings").count()
print(f"Delta table contains {count:,} records")

SUCCESS  written to dbw_weather_jm1535.sightline.canonical_sightings
Delta table contains 7,990 records


In [0]:
from pyspark.sql import functions as F

df = spark.table("dbw_weather_jm1535.sightline.canonical_sightings")

# 1. Time-series — sightings by month
timeseries_df = df.groupBy(
    F.date_trunc("month", F.col("observed_at")).alias("month")
).agg(
    F.count("*").alias("sighting_count"),
    F.countDistinct("species_scientific_name").alias("species_count"),
    F.sum("count").alias("total_individuals")
).orderBy("month")

timeseries_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_weather_jm1535.sightline.timeseries_monthly")

print(f"SUCCESS  timeseries_monthly — {timeseries_df.count()} months")

# 2. Species summary
species_df = df.groupBy(
    "species_common_name", "species_scientific_name", "kingdom", "family", "conservation_status"
).agg(
    F.count("*").alias("sighting_count"),
    F.sum("count").alias("total_individuals"),
    F.avg("latitude").alias("avg_latitude"),
    F.avg("longitude").alias("avg_longitude"),
    F.min("observed_at").alias("first_seen"),
    F.max("observed_at").alias("last_seen")
).orderBy(F.desc("sighting_count"))

species_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_weather_jm1535.sightline.species_summary")

print(f"SUCCESS  species_summary — {species_df.count()} species")

# 3. Heatmap grid — 0.5 degree cells
heatmap_df = df.groupBy(
    (F.round(F.col("latitude") / 0.5) * 0.5).alias("lat_cell"),
    (F.round(F.col("longitude") / 0.5) * 0.5).alias("lng_cell")
).agg(
    F.count("*").alias("sighting_count"),
    F.countDistinct("species_scientific_name").alias("species_count")
).orderBy(F.desc("sighting_count"))

heatmap_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_weather_jm1535.sightline.heatmap_grid")

print(f"SUCCESS  heatmap_grid — {heatmap_df.count()} grid cells")

SUCCESS  timeseries_monthly — 25 months
SUCCESS  species_summary — 255 species
SUCCESS  heatmap_grid — 1201 grid cells


In [0]:
display(spark.table("dbw_weather_jm1535.sightline.species_summary").limit(10))
display(spark.table("dbw_weather_jm1535.sightline.timeseries_monthly"))

species_common_name,species_scientific_name,kingdom,family,conservation_status,sighting_count,total_individuals,avg_latitude,avg_longitude,first_seen,last_seen
Southern Hairy-Nosed Wombat,Lasiorhinus latifrons,ANIMALIA,VOMBATIDAE,null,49,151,-32.25580892355346,144.12623512658465,2024-05-20T23:40:22.886802Z,2026-04-12T16:47:30.886802Z
Ratcheting Toadlet,Uperoleia stridera,ANIMALIA,MYOBATRACHIDAE,null,46,131,-28.66493547612665,144.1257363891001,2024-06-15T04:31:46.886802Z,2026-04-10T13:21:44.886802Z
null,Anthus,ANIMALIA,MOTACILLIDAE,null,44,128,-29.300047417584825,141.21837473828754,2024-05-01T15:59:52.886802Z,2026-04-18T13:35:34.886802Z
null,Pseudocolus,Fungi,Phallaceae,null,44,123,-31.81348475874718,144.70150747429037,2024-05-14T10:24:10.886802Z,2026-04-01T11:35:18.886802Z
Carnaby's Black-Cockatoo,Zanda latirostris,ANIMALIA,CACATUIDAE,null,44,100,-30.462254294418997,142.19501957643885,2024-04-29T09:05:17.886802Z,2026-04-15T14:05:41.886802Z
null,Zoothera,ANIMALIA,TURDIDAE,null,43,106,-31.648842910086373,138.56678831246035,2024-05-02T14:15:15.886802Z,2026-04-10T04:49:42.886802Z
null,Pagodroma,ANIMALIA,PROCELLARIIDAE,null,42,98,-27.699442325702133,139.42612994539775,2024-06-02T11:22:56.886802Z,2026-02-04T11:19:22.886802Z
null,Chaetothyrium,Fungi,Chaetothyriaceae,null,42,99,-30.874516412411406,140.51588247362872,2024-05-04T14:47:59.886802Z,2026-04-15T16:29:40.886802Z
Children's Python,Antaresia childreni,ANIMALIA,PYTHONIDAE,null,41,89,-29.676339137919474,141.3810267248821,2024-05-04T04:25:45.886802Z,2026-04-18T13:59:31.886802Z
null,Uperoleia,ANIMALIA,MYOBATRACHIDAE,null,41,150,-29.926957584156426,141.2479175931706,2024-06-01T06:37:23.886802Z,2026-03-31T18:07:09.886802Z


month,sighting_count,species_count,total_individuals
2024-04-01T00:00:00Z,61,52,174
2024-05-01T00:00:00Z,352,186,920
2024-06-01T00:00:00Z,327,182,1004
2024-07-01T00:00:00Z,329,177,1005
2024-08-01T00:00:00Z,350,190,1077
2024-09-01T00:00:00Z,332,190,938
2024-10-01T00:00:00Z,355,190,1091
2024-11-01T00:00:00Z,343,191,917
2024-12-01T00:00:00Z,366,202,1003
2025-01-01T00:00:00Z,330,183,902
